# Brain Tumor Pipeline Colab Training

This notebook trains the patient-safe two-stage pipeline on Google Colab. Use a GPU runtime: `Runtime > Change runtime type > T4 GPU`.

In [ ]:
!nvidia-smi
import os
import sys
from pathlib import Path
print(sys.version)

## 1. Get The Project

If you opened this notebook from inside the repository, leave `REPO_URL` blank. If you opened it as a standalone notebook, paste your GitHub repo URL below.

In [ ]:
REPO_URL = ""  # example: "https://github.com/YOUR_NAME/YOUR_REPO.git"
PROJECT_DIR = Path("/content/brain-tumor-pipeline")

if not (Path.cwd() / "src" / "brain_tumor_pipeline").exists():
    if not PROJECT_DIR.exists():
        if not REPO_URL:
            raise ValueError("Paste your GitHub repo URL into REPO_URL, then rerun this cell.")
        !git clone "$REPO_URL" "$PROJECT_DIR"
    os.chdir(PROJECT_DIR)

print("Project directory:", Path.cwd())
!ls

## 2. Install Dependencies

In [ ]:
!pip -q install -e .
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 3. Download Dataset From Kaggle

Upload your `kaggle.json` when prompted. Get it from Kaggle: Account > Create New API Token.

In [ ]:
DATA_ROOT = Path("/content/lgg-mri-segmentation")
DATASET_DIR = DATA_ROOT / "kaggle_3m"

if not DATASET_DIR.exists():
    from google.colab import files
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise ValueError("Upload kaggle.json to download the dataset.")
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json
    !mkdir -p "$DATA_ROOT"
    !kaggle datasets download -d mateuszbuda/lgg-mri-segmentation -p "$DATA_ROOT" --unzip

print("Dataset directory:", DATASET_DIR)
!find "$DATASET_DIR" -maxdepth 1 -type d | head

## 4. Dataset Sanity Checks

This forces metadata to be rebuilt from image/mask files, avoiding the known bad `patient_id` column in some copied CSVs.

In [ ]:
from brain_tumor_pipeline.data import load_metadata, group_train_val_test_split, assert_no_patient_overlap

MISSING_METADATA_CSV = Path("/content/no_data_mask_csv_here.csv")
df = load_metadata(metadata_csv=MISSING_METADATA_CSV, dataset_dir=DATASET_DIR)
train_df, val_df, test_df = group_train_val_test_split(df, validation_size=0.15, test_size=0.15, random_state=42)
assert_no_patient_overlap(train_df, val_df, test_df)

print("Rows:", len(df))
print("Patients:", df.patient_id.nunique())
print("Image-level class balance:")
print(df["mask"].value_counts(normalize=True).sort_index())
print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Patient split sizes:", train_df.patient_id.nunique(), val_df.patient_id.nunique(), test_df.patient_id.nunique())

## 5. Training Parameters

Safer defaults for Colab: train the classifier head first, then fine-tune only the top ResNet layers. Increase epochs after the smoke run looks healthy.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

STAGE = "classifier"  # "classifier", "segmenter", or "all"
CLASSIFIER_EPOCHS = 20
FINE_TUNE_EPOCHS = 10
SEGMENTER_EPOCHS = 60
FINE_TUNE_LAYERS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.45
LABEL_SMOOTHING = 0.03
USE_MIXED_PRECISION = True
OUTPUT_DIR = Path("/content/drive/MyDrive/brain_tumor_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Artifacts will be written to:", OUTPUT_DIR)

## 6. Train

For a quick check, set `CLASSIFIER_EPOCHS = 1` and `FINE_TUNE_EPOCHS = 0`.

In [ ]:
import subprocess

cmd = [
    sys.executable,
    "-m", "brain_tumor_pipeline.train",
    "--stage", STAGE,
    "--dataset-dir", str(DATASET_DIR),
    "--metadata-csv", str(MISSING_METADATA_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--batch-size", str(BATCH_SIZE),
    "--classifier-epochs", str(CLASSIFIER_EPOCHS),
    "--fine-tune-epochs", str(FINE_TUNE_EPOCHS),
    "--segmenter-epochs", str(SEGMENTER_EPOCHS),
    "--fine-tune-layers", str(FINE_TUNE_LAYERS),
    "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY),
    "--dropout", str(DROPOUT),
    "--label-smoothing", str(LABEL_SMOOTHING),
]
if USE_MIXED_PRECISION:
    cmd.append("--mixed-precision")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 7. Inspect Outputs

The trainer writes model checkpoints, CSV histories, dataset reports, metric JSON files, and overfitting reports.

In [ ]:
!find "$OUTPUT_DIR" -maxdepth 3 -type f | sort

import json
for report in sorted(OUTPUT_DIR.glob("**/*report*.json"))[:10]:
    print("\n---", report, "---")
    print(json.dumps(json.loads(report.read_text()), indent=2)[:2000])

## Recommended Runs

- Smoke test: `CLASSIFIER_EPOCHS = 1`, `FINE_TUNE_EPOCHS = 0`, `STAGE = "classifier"`.
- Classifier training: `CLASSIFIER_EPOCHS = 20`, `FINE_TUNE_EPOCHS = 10`, `FINE_TUNE_LAYERS = 50`.
- Segmenter training: `STAGE = "segmenter"`, `SEGMENTER_EPOCHS = 60`.
- If validation AUC/Dice gets worse while training improves, reduce fine-tune layers, reduce epochs, or increase dropout.